# Un-Language Slider — Coherence Dial Spike

Fine-tune an F5-TTS LoRA so a single control token (`tongues zero..four`) grades the intelligibility of the same input sentence in the same voice, from clean speech to phonotactically-valid English-native glossolalia. The fine-tune is the load-bearing trick: no prompt, no DSP plugin, and no shipped product (closed or open) does this — verified by three adversarial research workflows (originality landscape, Cocteau Twins frame audit, glossolalia-converter prior-art sweep).

**Gates that must ALL pass before we ship:**
- Whisper-WER rises monotonically across levels 0..4, Spearman ≥ +0.80 (positive sign — WER *rises* as dial rises).
- Resemblyzer cosine vs the level-0 reference stays ≥ 0.85 (voice preserved).
- Perceptual gate: hand-listen `dial=2` wavs — must be a partial dissolution, not bimodal collapse.

**Recommended order:** run the **200-clip micro-spike** (Cell 3 default, ~17 min on A100) first to confirm the LoRA learns the control token at all. If the micro-spike's Cell-7 verdict is PARTIAL or PASS, scale to the full 7,500-clip run before committing to the long training.

Runtime: A100/H100 ZeroGPU strongly recommended (F5-TTS ~5-7s per generated clip).

In [ ]:
# Cell 1 — install F5-TTS + validation deps + repo scripts
!nvidia-smi -L
!apt-get -qq install -y ffmpeg >/dev/null
!pip install -q f5-tts g2p_en jiwer openai-whisper resemblyzer huggingface_hub pedalboard soundfile librosa scipy numpy peft >/dev/null
# Bring in the repo scripts. If you renamed the repo, adjust REPO_URL.
REPO_URL = 'https://github.com/akshan-main/glossolalia.git'
import os
if not os.path.isdir('/content/repo'):
    !git clone -q $REPO_URL /content/repo || echo 'clone failed - upload scripts/ manually via Files panel'
%cd /content/repo
!ls scripts/ | head -20
print('install done')

In [ ]:
# Cell 2 — pull data inputs (sentences + voice refs + phoneme LM) from HF
from huggingface_hub import snapshot_download
import shutil, os
REPO = 'akshan-main/glossolalia-inputs'   # populated by scripts/push_data_inputs.py before running this notebook
p = snapshot_download(repo_id=REPO, repo_type='dataset', local_dir='/content/repo/data_pull')
# Stage into the layout the scripts expect (data/sentences.txt, data/voices/v*.wav, data/phoneme_lm.npz)
os.makedirs('/content/repo/data', exist_ok=True)
os.makedirs('/content/repo/data/voices', exist_ok=True)
for f in ['sentences.txt', 'phoneme_lm.npz', 'cmudict.dict']:
    src = os.path.join(p, f)
    if os.path.exists(src): shutil.copy(src, f'/content/repo/data/{f}')
voices_dir = os.path.join(p, 'voices')
if os.path.isdir(voices_dir):
    for f in os.listdir(voices_dir):
        shutil.copy(os.path.join(voices_dir, f), f'/content/repo/data/voices/{f}')
print('  sentences:', len(open('/content/repo/data/sentences.txt').readlines()))
print('  voices   :', sorted(os.listdir('/content/repo/data/voices')))
assert os.path.exists('/content/repo/data/phoneme_lm.npz'), 'phoneme_lm.npz missing'

In [ ]:
# Cell 3 — generate the training corpus on GPU (F5-TTS base, no LoRA yet)
# Scales:
#   micro-spike   ->  40 sentences  x 5 levels x 1 voice =   200 clips  (~17 min on A100)
#   single-voice  -> 500 sentences  x 5 levels x 1 voice = 2,500 clips  (~3-4 h)
#   full          -> 500 sentences  x 5 levels x 3 voices = 7,500 clips (~10-12 h)
# Recommended: run the 200-clip micro-spike FIRST. Train a tiny LoRA on it (Cell 5 with --epochs 1)
# and read Cell 7's verdict. If Spearman > 0.5, scale up. If 0, the corruption procedure or token
# format needs fixing BEFORE you spend 10 hours.
!python scripts/generate_coherence_data.py \
  --sentences data/sentences.txt \
  --voice v1:data/voices/v1.wav:data/voices/v1.txt \
  --lm data/phoneme_lm.npz \
  --out data/coherence \
  --max-sentences 40 \
  --levels 5 \
  --input-mode pseudo \
  --resume
# To scale: add `--voice v2:data/voices/v2.wav:data/voices/v2.txt --voice v3:data/voices/v3.wav:data/voices/v3.txt`
# and bump `--max-sentences 500`.

In [ ]:
# Cell 4 — build the F5-TTS finetune dataset (CSV + JSONL + symlinked wavs)
!python scripts/build_coherence_dataset.py --data data/coherence --out data/coherence_ds
!head -3 data/coherence_ds/metadata.csv
!wc -l data/coherence_ds/metadata.csv

In [ ]:
# Cell 5 — LoRA finetune on F5-TTS
# Per TTS-choice research verdict:
#   - F5-TTS was chosen DELIBERATELY because it's a flat character-to-mel model with no LLM
#     front-end, so it pronounces our corrupted pseudo-spelling as written (no auto-correct).
#     If you swap to Spark/Chatterbox/Orpheus/Higgs the LLM backbone will normalize "kah tivahchhin"
#     toward real English and flatten the dial — don't.
#   - LoRA wraps the DiT q/k/v/out_proj linears. If the official CLI's --lora_rank arg isn't in
#     your installed version, drop it and use a community fork (e.g. lpscr/F5-TTS-Finetune-with-LoRA)
#     which exposes the PEFT wrapper directly.
# Key flags: rank 16, alpha 16, bf16, micro-spike epochs 1 (sanity), full run epochs 5.
EPOCHS = 1   # micro-spike: 1 epoch on 200 clips ~ minutes. Set to 5 for the full 7,500-clip run.
!f5-tts_finetune-cli \
  --exp_name unlang_lora \
  --tokenizer_path data/coherence_ds/metadata.csv \
  --dataset_name coherence_ds \
  --learning_rate 1e-5 \
  --batch_size_per_gpu 4 \
  --batch_size_type sample \
  --max_samples 64 \
  --grad_accumulation_steps 2 \
  --max_grad_norm 1.0 \
  --epochs $EPOCHS \
  --num_warmup_updates 200 \
  --save_per_updates 3000 \
  --last_per_steps 6000 \
  --finetune True \
  --tokenizer pinyin \
  --logger tensorboard \
  --lora_rank 16 --lora_alpha 16 || echo 'CLI signature drift — try the community LoRA fork:'
import glob
ckpts = sorted(glob.glob('/content/repo/**/unlang_lora*/checkpoints/*.safetensors', recursive=True))
print('checkpoints:', ckpts[-3:] if ckpts else 'NONE')

In [ ]:
# Cell 6 — sweep the dial: 10 holdout sentences x N voices x 5 levels x 3 seeds
import glob
lora_ckpts = sorted(glob.glob('/content/repo/**/unlang_lora*/checkpoints/*.safetensors', recursive=True))
assert lora_ckpts, 'no LoRA checkpoint — Cell 5 must finish before Cell 6'
LORA = lora_ckpts[-1]; print('using', LORA)
!python scripts/sweep_dial.py \
  --lora $LORA \
  --voices v1:data/voices/v1.wav:data/voices/v1.txt \
  --out sweep \
  --levels 5 \
  --seeds 42,123,7 \
  --max-sentences 10
# Once micro-spike passes, scale to multi-voice:
# --voices v1:data/voices/v1.wav:data/voices/v1.txt,v2:data/voices/v2.wav:data/voices/v2.txt,v3:data/voices/v3.wav:data/voices/v3.txt

In [ ]:
# Cell 7 — validate: Whisper WER (monotonic + Spearman ≥ +0.80) + Resemblyzer cosine (≥ 0.85)
!python scripts/evaluate_coherence_dial.py \
  --manifest sweep/sweep_manifest.json \
  --out sweep/eval_report.json \
  --whisper base.en \
  --spearman-gate 0.80 \
  --cosine-gate 0.85

In [ ]:
# Cell 8 — download dial=0/2/4 wavs for the human perceptual gate
from google.colab import files
import glob
picks = []
for lv in (0, 2, 4):
    g = sorted(glob.glob(f'sweep/v1_lv{lv}_s42_*.wav'))
    if g: picks.append(g[0])
for p in picks:
    print(p); files.download(p)

## Reading the result

- **All gates pass** (Spearman ≥ +0.80, min cosine ≥ 0.85, dial=2 perceptually partial) → push the LoRA + sweep wavs, build the toy.
- **Micro-spike Spearman 0** → the LoRA isn't learning the token. Check: is `tongues four` being split into chars (look at training logs)? Did F5-TTS swallow the `|` separator? Try the community LoRA fork.
- **Micro-spike Spearman 0.3-0.6** → real signal, just under-trained. Scale to full 7,500-clip run + 5 epochs.
- **WER non-monotonic** at full scale → tighten Markov LM to a vowel-heavy / sonorant-consonant palette and retrain.
- **Voice cosine < 0.85 at high levels** → lower rank to 8 + restrict target_modules to text-cross-attn only.
- **Dial=2 bimodal** → flatten the p-schedule (e.g., `[0, 0.20, 0.45, 0.70, 0.95]`) and curriculum-finetune on level-2-weighted mini-batches.

Push the trained LoRA:
```python
from huggingface_hub import HfApi
HfApi().upload_folder(folder_path='/content/repo/ckpts/unlang_lora', repo_id='akshan-main/glossolalia-dial-lora', repo_type='model')
```